In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import tifffile
import dask.array as da
import utils
import skimage as ski
import pandas as pd
# import time
# import extract_features
# import ast
# from cellpose.models import CellposeModel
# from cellpose.io import logger_setup
# logger_setup()
import dask_image.ndfilters as dask_ndi
from deeptile.extensions import stitch
import deeptile
import importlib
importlib.reload(utils)
import brieflow_segment_utils
importlib.reload(brieflow_segment_utils)
from brieflow_segment_utils import image_log_scale, reconcile_nuclei_cells, dask_image_log_scale, tiled_reconcile_nuclei_cells
from deeptile.core.lift import lift
from collections import defaultdict

In [ ]:
img_path = "/Users/hannahbolen/Desktop/image_analysis/slide_tiff/o8p_day24_s12.ome.tif"
coverslip_path = "/Users/hannahbolen/Desktop/image_analysis/masks_and_results/o8p_day24_s12_coverslip.tif"
save_path = "/Users/hannahbolen/Desktop/image_analysis/masks_and_results/o8p_day24_s12_cytoplasm_mask.tif"
img_dtype = tifffile.TiffFile(img_path).pages[0].dtype

# coverslip = da.from_zarr(tifffile.imread(coverslip_path, aszarr=True))
# img = da.from_zarr(tifffile.imread(img_path, aszarr=True))*coverslip
nuclei_mask = da.from_zarr(tifffile.imread("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/o8p_day24_s12_nuclei_mask.tif", aszarr=True))
cyto_mask = da.from_zarr(tifffile.imread(save_path, aszarr=True))

In [ ]:
# preprocessing
img0 = img[0].copy()
img1 = img[1].copy()

# log transform and normalize GFP channel
img0 = da.log(da.where(img0 == 0, 0.01, img0))
x01 = da.percentile(img0, 0.1, axis=(0, 1)).compute()
x99 = da.percentile(img0, 99, axis=(0, 1)).compute()
img0 = (img0 - x01) / (x99 - x01)

In [ ]:
# log transform cy5 w different method
img1 =  dask_image_log_scale(img1)

# stack and gaussian blur
img = da.stack([img0, img1])
img = dask_ndi.gaussian_filter(img, sigma=(0,2,2))
img = img*coverslip

In [2]:
nuclei_path = "/Users/hannahbolen/Desktop/image_analysis/masks_and_results/o8p_day24_s12_filtered_nuclei.tif"
cyto_path = "/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/reconcile_o8p_day24_s12_cyto_mask.tif"

nuclei_mask = da.from_zarr(tifffile.imread(nuclei_path, aszarr=True))
cyto_mask = da.from_zarr(tifffile.imread(cyto_path, aszarr=True))

In [3]:
tile_size = (2048, 2048)
overlap = (0.1, 0.1)

nuclei_dt = deeptile.load(nuclei_mask)

nuclei_tiles = nuclei_dt.get_tiles(tile_size, overlap)
cyto_tiles = nuclei_tiles.import_data(cyto_mask, "image").unpad()

In [ ]:
nuclei, cells = tiled_reconcile_nuclei_cells(nuclei_tiles,cyto_tiles)

In [4]:
nuclei, cells = tiled_reconcile_nuclei_cells(nuclei_tiles,cyto_tiles)
reconcile_nuclei = stitch.stitch_masks(nuclei)
reconcile_cells = stitch.stitch_masks(cells)
tifffile.imwrite("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/reconcile_o8p_day24_s12_cyto_mask.tif", reconcile_cells.astype(img_dtype))
tifffile.imwrite("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/reconcile_o8p_day24_s12_nuclei_mask.tif", reconcile_nuclei.astype(img_dtype))

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------------

--------------------

NameError: name 'img_dtype' is not defined

In [4]:
nuclei, cells = reconcile_nuclei_cells(nuclei_mask, cyto_mask, how='contained_in_cells')
tifffile.imwrite("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/reconcile_o8p_day24_s12_cyto_mask.tif", cells.astype(img_dtype))
tifffile.imwrite("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/reconcile_o8p_day24_s12_nuclei_mask.tif", nuclei.astype(img_dtype))

KeyboardInterrupt: 

In [ ]:
model_parameters = {'gpu': True}
eval_parameters = {'diameter':100, 'batch_size':16, 'normalize':False}
# model = CellposeModel(**model_parameters)
# cyto_mask, _, _ = model.eval(img, **eval_parameters)
cellpose = utils.cellpose_segmentation(model_parameters, eval_parameters, threshold=0)
masks_cyto = cellpose(img_tiles)
mask_cyto = stitch.stitch_masks(masks_cyto)

tifffile.imwrite(save_path, mask_cyto.astype(img_dtype))
# fig, ax = plt.subplots(figsize=(20,20))
# ax.imshow(ski.segmentation.mark_boundaries(img_log[1], cyto_mask, color=(1,0,1)))
# ax.set_yticks([])
# ax.set_xticks([])
# plt.tight_layout()


In [ ]:
hasattr(cyto_tiles[0,0], "compute")

In [ ]:
# fig, ax = plt.subplots(ncols=2,figsize=(20,10))
fig, ax = plt.subplots(figsize=(10,10))

# ax[0].imshow(ski.segmentation.mark_boundaries(ski.segmentation.mark_boundaries(ski.exposure.rescale_intensity(img[0], in_range=(0,4000)), nuclei, color=(1,0,0), mode='thick'), cells, color=(1,0,1)))
# ax[1].imshow(ski.segmentation.mark_boundaries(ski.segmentation.mark_boundaries(img_log[1], nuclei, color=(1,0,0), mode='thick'), cells, color=(1,0,1)), cmap='gray')


# # # ax[1].imshow(ski.color.label2rgb(mask))
# # # ax[1].set_yticks([])
# # # ax[1].set_xticks([])
# # # plt.tight_layout()


# # # # #img2 = da.from_zarr(tifffile.imread(img_path, aszarr=True))[1,y0:y1,y0:y1]

# # # cy5 = da.from_zarr(tifffile.imread(img_path, aszarr=True))[1,y0:y1,y0:y1]
# # # ax.imshow(ski.segmentation.mark_boundaries(ski.exposure.rescale_intensity(cy5, in_range=(0,4000)), mask, color=(1,0,1), mode='thick'), cmap='gray')
ax.imshow(masks_cyto[9,17], cmap='gray')


# ax[0].imshow(img_tiles[9,7][0], cmap='gray')
# ax[1].imshow(img_tiles[9,7][1], cmap='gray')


ax.set_yticks([])
ax.set_xticks([])
# ax[0].set_yticks([])
# ax[0].set_xticks([])
# ax[1].set_yticks([])
# ax[1].set_xticks([])

plt.tight_layout()